[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Crew/transport-networks-lab/blob/main/Notebooks/Practica_guiada1_IntroNx.ipynb)


#  Grafos y redes de transporte

##  Parte 1. Teoría de grafos con `NetworkX`
### Elementos iniciales

#### Algunos conceptos básicos

A modo de inicio, cabe aclarar que no siempre existe una forma única de representar una red. Mientras esta refiere al funcionamiento de un sistema complejo, un `grafo` es la representación matemática de dicha red de relaciones. En ese sentido, contamos con infinidades de esquemas. Dentro de ellos las redes de transporte son uno más, donde generalmente las estaciones se representan por medio de nodos o vértices, y las relaciones entre ellos a través de calles o viajes. De tal manera, un grafo se compone por:

<img align="center" width="700" height="500" src="https://github.com/PyMap/AUPY/blob/master/imagenes/nodos_ejes.png?raw=1" style="float: center; padding: 0 15px">

Para estar al tanto de la última versión disponible siempre es recomendable chequear en el [repositorio oficial](https://github.com/networkx/networkx). También dejamos el link a la [documentación](https://networkx.org/documentation/stable//reference/introduction.html) de la librería.

In [ ]:
from google.colab import drive
drive.mount('/drive/')

In [ ]:
# última release
!pip show networkx | grep Version

## Sección 1: Armando nuestro grafo

Comencemos por armar un grafo no dirigido, distinguiendo nodos y ejes.

In [ ]:
# importar librerias
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
# creamos un grafo no dirigido
G = nx.Graph()

In [ ]:
dir(G)

In [ ]:
# armamos una lista con los que serán nuestros nodos
nodos = list(range(9))

In [ ]:
# y la usamos para agregarlos a nuestro objeto de tipo Graph
G.add_nodes_from(nodos)

Como vimos, este objeto se crea cuando instanciamos la clase Graph. Tanto los nodos como los ejes son atributos de este tipo de objetos. Veamos cómo se puede acceder a los mismos.

In [ ]:
# los nodos son un atributo de los grafos. Nosotros creamos 9 en total
G.nodes()

Ahora definamos los ejes de nuestro grafo. O en otras palabras, **de qué manera vamos a conectar los nodos:**

In [ ]:
print('Conectado nodos:')
print('***************')
for n in range(0, len(nodos),3):
    print(n, '-', n+1)
    G.add_edge(n, n+1)
    print(n+1, '-', n+2)
    G.add_edge(n+1, n+2)

In [ ]:
# y los devolvemos como una lista de tuplas. Cada una, con los nodos que conecta el eje
G.edges()

Usemos el método `draw` de NetworkX para previsualizar cómo se vería esta red de relaciones

In [ ]:
nx.draw(G)

Vale aclarar que todavía no exploramos ninguno de los parametros de `draw`. Tampoco agregamos la noción de espacialidad a nuestro grafo, con lo cual, al no fijar la posición de los nodos este método asigna aleatoriamente su ubicación cada vez que lo ejecutamos:

In [ ]:
# conectemos ahora los segmentos inconexos para conformar toda la red
G.add_edge(2,5)
G.add_edge(7,5)

In [ ]:
# vemos que las relaciones se mantienen, pero la forma del grafo cambia
nx.draw(G)

A lo largo de este notebook iremos viendo cuáles son las posibilidades que nos brinda este método para poder customizar nuestro grafo. Prosigamos...

### 1.1. Nodos o vértices

Como dijimos, uno de los componentes principales de un grafo son sus nodos. Ya sabemos que con el método `add_nodes_from` podemos incluirlos en nuestro objeto de tipo Graph. Ahora veamos qué tipo de operaciones podemos hacer sobre los mismos.

Veremos que los nodos se vuelven accesibles, como cualquier iterable, a través de un índice. Una vez allí, podemos decidir qué tipo de atributo agregarle con una etiqueta. Tal como si fueran diccionarios.

In [ ]:
# agregamos al nodo 1 una label que vamos a definir con un nombre, suponiendo que fuese una estación.
G.nodes[0]['estacion'] = 'CALLAO'

# también podemos agregar la línea
G.nodes[0]['linea'] = 'B'

# Algún color para identificar
G.nodes[0]['color'] = 'red'

# y por qué no sus coordenadas
G.nodes[0]['coord'] = (-58.39241446187341, -34.603705218144896)

Como se puede apreciar, agregamos una etiqueta a uno de nuestros nodos. De esta manera, podemos obtener una lista de nodos con la metadata asociada. Si prestamos atención, el parámetro `data` nos devuelve un diccionario donde la key es el nodo. Despues contamos con la metadata en otro diccionario donde la llave es la etiqueta y la key el valor.

Ahora sólo contamos con una descripción a modo de ejemplo, pero imaginen que dicho valor podría ser la cantidad de pasajeros arrivados a una estación a una hora determinada, o el promedio de tiempo entre una estación y otra.

In [ ]:
# Veamos el parametro data
G.nodes(data=True)

In [ ]:
# y accedamos a su metadata con el posicional de la tupla donde se encuentra el nodo y el diccionario
list(G.nodes(data=True))[0][1]

In [ ]:
G.name = 'Subtes CABA'

In [ ]:
G.name

Completemos nuestra red con la metadata de los demás nodos ...

In [ ]:
import pandas as pd

def atributos_para_nodos():
    '''
    Prepara un diccionario con atributos de algunas estaciones del Subte de la CABA
    '''

    # cargamos el dataframe y lo filtramos con nuestros nodos
    subte = pd.read_csv('/drive/MyDrive/Técnicas y Análisis de datos del Transporte/Data/estaciones-de-subte.csv')
    lineas = subte[subte.linea.isin(['C','B','D'])]

    estaciones = lineas[lineas.estacion.isin(['URUGUAY', 'C. PELLEGRINI',
                                              '9 DE JULIO', 'CALLAO',  'TRIBUNALES - TEATRO COLÓN',
                                              'AV. DE MAYO', 'DIAGONAL NORTE', 'LAVALLE'])].copy()

    # agrupamos las coordenadas
    estaciones['coord'] = list(zip(estaciones.long, estaciones.lat))

    # restablecemos el orden en función de cómo contectamos los nodos de nuestro graFo
    estaciones = estaciones.sort_values(by=['linea','id'], ascending=False)
    estaciones_reorder = []
    for l in ['B','D','C']:
        estaciones_reorder.append(estaciones[estaciones.linea==l])

    estaciones = pd.concat(estaciones_reorder).reset_index().drop(columns='index')

    # ahora asociacmos cada estación a los nodos del grafo
    estaciones['node_idx'] = list(range(1, len(estaciones)+1))

    # creamos el resto de los atributos, reindexamos y borramos las columnas que no nos sirven
    estaciones['color'] = estaciones.linea.replace({'D':'green', 'C':'blue', 'B':'red'})
    estaciones.set_index(estaciones.estacion, inplace=True)
    estaciones.drop(columns=['estacion','id', 'long', 'lat'], inplace=True)

    atributos = estaciones.to_dict(orient='index')

    return atributos

input_dict = atributos_para_nodos()


In [ ]:
# Asi luce nuestro diccionario de atributos
input_dict

In [ ]:
def define_atributos_nodos(atributos, nombre_estacion):
    '''
    Utiliza un diccionario de atritbutos para poblar los nodos de un grafo
    '''
    # agregamos al nodo 1 una label que vamos a definir con un nombre, suponiendo que fuese una estación.
    G.nodes[atributos[nombre_estacion]['node_idx']]['estacion'] = nombre_estacion

    # también podemos agregar la línea
    G.nodes[atributos[nombre_estacion]['node_idx']]['linea'] = atributos[nombre_estacion]['linea']

    # Algún color para identificar
    G.nodes[atributos[nombre_estacion]['node_idx']]['color'] = atributos[nombre_estacion]['color']

    # y por qué no sus coordenadas
    G.nodes[atributos[nombre_estacion]['node_idx']]['coord'] = atributos[nombre_estacion]['coord']

In [ ]:
for k in input_dict.keys():
    define_atributos_nodos(atributos=input_dict, nombre_estacion=k)

In [ ]:
G.nodes(data=True)

In [ ]:
# accedemos a la metadata del nodo 3 y vemos si se asignó la información que esperabamos
list(G.nodes(data=True))[3][1]

Supongamos ahora que quisieramos quedarnos solamente con aquellos nodos que responden a un atributo en particular. Por ejemplo, pertenecer a una misma línea del subterráneo. Se les ocurre cómo podríamos filtrar elementos dentro de la vista de nodos de un grafo determinado?

In [ ]:
# Una recurso bastante utilizado en este tipo de estructuras son las comprehension lists!
[n for n,d in G.nodes(data=True) if d['linea'] == 'B']

Ya sabemos que los nodos `0`, `1` y `2` pertenecen a la línea `B`.

### 1.2. Ejes o arcos

Como mencionamos, otro de los componentes de un grafo son los ejes. Estos, establecen o determinan si los pares de nodos que conforman nuestro grafo están conectados por algún tipo de relación. De manera similar a como vimos recién, los ejes se vuelven accesibles mediante un indexado. Ahora bien, este no es un valor único sino que considera los extremos del eje. En otras palabras, los nodos que está conectando.

In [ ]:
# Agreguemos ahora un par "key:value" a los ejes. Vean que para acceder se usa una lista con los nodos que unen
G.edges[0,1]['fecha']='2019-10-01'
G.edges[0,1]['hora']='09:30:00'
G.edges[0,1]['pasajeros']=125

In [ ]:
G.edges(data=True)

Así, acabamos de agregar un poco de información adicional a nuestra primera conexión. Los nodos `0` y `1` se conectaban el 10 de enero de 2019 a las 9:30 de la mañana, transportando 125 pasajeros. Esto nos brinda algo de contexto.

Ahora bien, si nos detenemos en este último atributo vamos a incorporar un nuevo concepto: el de peso (o `weight` en inglés). Este sirve para dimensionar la intensidad de una conexión entre pares de nodos dentro de nuestro grafo. Y así como decidimos que sean pasajeros, podríamos haber definido cualquier otro: tiempo transcurrido para llegar de un nodo `u` a otro `v`, velocidad promedio del viaje, etc. etc.

Al igual que con los nodos, agregar atributos a los ejes nos permite caracterizar a nuestro grafo. Y también poder seleccionar componentes dentro del mismo siguiendo algún tipo de condición. Veamos cómo hacerlo iterando sobre los ejes de una manera más extendida, pero también más explícita ...

In [ ]:
# para los nodos o vértices (u,v) y el diccionario (d)
for u,v,d in G.edges(data=True):

    # revisamos si el primer nodo se encuentra involucrado:
    if 0 in [u,v]:

        # y le cambiamos el valor de su peso
        d['pasajeros'] = 75

In [ ]:
# chequeamos que solo actulizamos el primer eje
[(u,v,d) for u,v,d in G.edges(data=True) if 1 in [u,v]]

In [ ]:
# mas simple
G.edges[0,1]

Ahora, completemos nuestro grafo sumando atributos al resto de los ejes. Para mantenerlo simple, supongamos que estamos trabajando con un recorte de la red de subterráneos para un mismo día y hora.

In [ ]:
import random

In [ ]:
def define_atributos_ejes():
    '''
    Define los atributos de los ejes de las tres lineas
    '''
    # estos son los nodos que conectan la lineas del subterraneo
    conexiones_lineas = [(2,5),(5,7)]
    conexiones_estaciones = [i for i in G.edges if i not in conexiones_lineas]

    # describimos las conexiones a lo largo de una misma linea
    for i in conexiones_estaciones:
        # donde i[0] es el nodo inicial y i[1] el final
        G.edges[i[0], i[1]]['fecha']='2019-10-01'
        G.edges[i[0], i[1]]['hora']='09:30:00'
        G.edges[i[0], i[1]]['pasajeros']=random.randint(10,100)

    # describimos las conexiones entre lineas
    for i in conexiones_lineas:
        G.edges[i[0], i[1]]['fecha']='2019-10-01'
        G.edges[i[0], i[1]]['hora']='09:30:00'
        G.edges[i[0], i[1]]['pasajeros']=random.randint(100,200)

In [ ]:
# asigamos atributos a los ejes
define_atributos_ejes()

In [ ]:
# y los visualizamos
G.edges(data=True)

In [ ]:
G.edges[6,7]

Por último, veamos cómo funciona el filtrado por listas de comprenión que tambien vimos con los nodos. Fíjense que para iterar sobre los ejes debemos tener en cuenta los nodos que están uniendo. Por eso la iteración es a partir de una tupla de elementos.

In [ ]:
# usamos una lista por comprensión para filtrar las conexiones con más de 75 pasajeros
[(u,v) for u,v,d in G.edges(data=True) if d['pasajeros'] > 75]

### 1.3. Visualizando nuestro grafo

Cuando visualizamos nuestro grafo `G` por primera vez, vimos que la posición de los nodos cambiaba cada vez que instanciábamos el objeto. Eso, porque no habíamos incorporado todavía la noción de espacialidad dentro del grafo. Esto, podemos hacerlo a través del parámetro `position` o `pos`. En el dataframe que usamos para articular los nodos contábamos con las coordenadas, aprovechemos ese atributo para ordenar las estaciones dentro del grafo, tal como si estuviésemos trabajando con la red de subterráneos.

In [ ]:
# el método `get_node_attributes` nos permite capturar los atributos desde un grafo ya instanciado
pos = nx.get_node_attributes(G,'coord')

In [ ]:
# y nos devuelve un diccionario con el idx del nodo y el atributo en cuestión
pos

In [ ]:
# que en nuestro grafo usamos para posicionar los nodos. Agreguemos también algo de color ...
nx.draw(G,pos, with_labels=True, node_color='#00b4d9')

Comparen este grafo con la red de subterráneos de la Ciudad de Buenos Aires. Ya empieza a haber un poco más de similitud no? Agreguemos algunos estilos adicionales para terminar de darle forma ...

In [ ]:
# primero con el nombre de las estaciones
lab = nx.get_node_attributes(G,'estacion')

In [ ]:
nx.draw(G,pos, labels=lab, with_labels=True, node_size=400, node_color='#00b4d9')

Ya con el nombre de las estaciones podemos ver qué es lo que estamos queriendo representar. Un grafo donde las líneas están representadas por estaciones que se unen consecutivamente, pero que a la vez, cuentan con ejes que las conectan entre sí. Evitando que las líneas se encuentren aisladas y que el grafo termine de completarse. Veamos cómo podríamos hacer para asignarle un color a los nodos en función de su pertenencia a cada una de las líneas.

In [ ]:
# podríamos seguir usando el método `get_node_attributes`
color_dict = nx.get_node_attributes(G,'color')

In [ ]:
# lo que hace el camino un poco más largo
colors = []
for n,c in color_dict.items():
    colors.append(c)

In [ ]:
colors

In [ ]:
# o tambien haber usado una comprehension list
colores = [i[1]['color'] for i in G.nodes(data=True)]

In [ ]:
colores

In [ ]:
nx.draw(G,pos, labels=lab, with_labels=True, node_size=400, node_color=colors)

Tanto los colores como los pesos se asignan por medio de listas, respetando el orden de los componentes (nodos o ejes) dentro del grafo. Veamoslo cómo sería aplicado a la medida de peso que agregamos recientemente, la cantidad de pasajeros. Veamos cómo queda.

In [ ]:
# asignamos el color amarillo a las conexiones intensas
ec = ['yellow' if G[u][v]['pasajeros'] > 100 else 'lightgrey' for u,v in G.edges()]

In [ ]:
# y definimos el parametro peso a partir del atributo pasajeros para visualizar
ew = [G[u][v]['pasajeros']/5 for u,v in G.edges()]

In [ ]:
# con el método `subplots` de matplotlib creamos el eje y la figura donde se almacena
fig, ax = plt.subplots(figsize=(17,10))

# dicho eje lo podemos pasar como parámetro al método `draw`
nx.draw(G,pos, labels=lab, with_labels=True,
        node_size=400, node_color=colors,
        edge_color = ec, width = ew, ax=ax)

# agregamos etiquetas a los ejes
edge_labels = nx.get_edge_attributes(G, 'pasajeros')
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8)

ax.set_title('Cantidad de pasajeros por eje');

También podríamos graduar los ejes con una leyenda que indique la cantidad de pasajeros, en lugar de usar una etiqueta. Esto con el módulo `cm` de matplotlib.

In [ ]:
from matplotlib import cm

In [ ]:
fig, ax = plt.subplots(figsize=(17,10))

ec=[G[u][v]['pasajeros'] for u, v in G.edges]
mcl = nx.draw_networkx_edges(G, pos, edge_cmap=cm.Greys, width=10,edge_color=ec)

nx.draw_networkx_nodes(G, pos, node_size=400, node_color=colors, ax=ax)
nx.draw_networkx_labels(G, pos, labels=lab, font_size=10,
                        horizontalalignment='right', ax=ax)

plt.colorbar(mcl)
ax.set_axis_off()
ax.set_title('Cantidad de pasajeros por eje');

### 1.2. Tipos de grafos

En el grafo que venimos construyendo como ejemplo no hemos incluído una cuestión central. Si bien pudimos diferenciar nodos a partir de conexiones representadas por cantidad de pasajeros, nunca especificamos en qué dirección se daba tal magnitud.

Dado que estamos representado viajes en una red de subterráneos, los pasajeros podrían haber ido de una estación a otra, o vicebersa. Introducimos así otra cuestión de mucha relevancia a la hora de trabajar con redes de transporte: `la direccionalidad`

`NetworkX` [cuenta con varias clases](https://networkx.org/documentation/stable//reference/classes/index.html) para poder representar el sentido de las conexiones entre nodos. En este sentido, es importante aclarar que, a grandes rasgos, contamos con dos familias de grafos. Los dirigidos y los no dirigidos:

<img align="center" width="900" height="400" src="https://github.com/PyMap/AUPY/blob/master/imagenes/grafos_tipo.png?raw=1" style="float: center; padding: 0 15px">

In [ ]:
# Veamos qué tipo de grafo es el que construimos previamente
type(G)

Tal como se puede apreciar, este es un grafo no dirigido. Es decir, ningún tipo de direccionalidad se ve representada. O al menos no es explícita, con lo cual la conexión entre nodos se da en ambos sentidos de la red.

Ahora, antes de seguir con nuestro ejemplo, exploremos brevemente qué alternativas tenemos para representar relaciones en distintos tipos de grafos con `NetworkX`.

#### 1.2.1. Grafos dirigidos

La principal característica de un grafo dirigido radica en el sentido de la relación entre nodos. Algo bastante central en las redes de transporte. La clase `DiGraph` de NetworkX permite construir grafos con ejes dirigidos, con la única restricción de no incluir [ejes paralelos o múltiples](https://es.wikipedia.org/wiki/Aristas_m%C3%BAltiples#:~:text=En%20teor%C3%ADa%20de%20grafos%2C%20las,m%C3%BAltiples%20son%20llamados%20grafos%20simples.).

Existen varias maneras de poblar un objeto de tipo `DiGraph`, ya sea agregando nodos singularmente o a partir de colecciones como listas. También es posible especificar directamente los ejes como mostramos a continuación:

In [ ]:
# Grafo dirigido
D = nx.DiGraph()
D.add_edge(0,1)
type(D)

In [ ]:
nx.draw(D, with_labels=True)

Tal y como mencionamos, este tipo de grafos no soportan los ejes paralelos. Comprobémoslo ...

In [ ]:
# repetimos algunos ejes
D.add_edges_from([(0,1),(0,1), (0,1), (1,0)])

In [ ]:
# al no ser un multigrafo, las conexiones entre el mismo par de nodos se cuentan sólo una vez!
D.number_of_edges(u=0,v=1)

Así, contamos en total con ...

In [ ]:
# dos conexiones (contabilizadas de manera unitaria, es decir, sin tener en cuenta cuantas veces suceden)
D.number_of_edges()

Porque recordemos, es un grafo dirigido. Entonces, la direccionalidad importa ...

In [ ]:
# no es lo mismo ir de 0 a 1 que de 1 a 0
D.number_of_edges(u=1,v=0)

In [ ]:
# graficamente se entiende mejor
nx.draw(D, with_labels=True, arrows=True, connectionstyle='arc3, rad = 0.1')

La intención de este segmento no es adentrarnos en detalle en los tipos de grafos. Sino más bien presentarlos y tener una idea del tipo de relaciones que cada uno permite representar. A los que les interese ahora un poco más, la documentación oficial es bastante clara y accesible. En el siguiente hipervínculo podrán encontrar más sobre [grafos dirigidos](https://networkx.org/documentation/stable//reference/classes/digraph.html#networkx.DiGraph)

#### 1.2.2. Multigrafos o grafos multiejes

Como primer gran distinción, en teoría de grafos podemos hablar de `grafos simples` o `multigrafos`. [Estos últimos](https://es.wikipedia.org/wiki/Multigrafo) pueden tener más de un eje o arista conectando los mismos nodos (lo que dijimos, nuestro `D` no soportaba). Algo bastante útil si pensamos, por ejemplo, en una red de calles donde los nodos estén representados por el cruce de las mismas.

Para este tipo de situaciones, NetworkX cuenta con la clase `MultiGraph` que permite trabajar con conexiones no dirigidas entre nodos relacionados por más de una arista. [Acá](https://networkx.org/documentation/stable//reference/classes/multigraph.html#networkx.MultiGraph) el link a la documentación.

In [ ]:
# Instanciamos un grafo no dirigido con ejes múltiples
M = nx.MultiGraph()
type(M)

In [ ]:
# y lo poblamocs con nodos y ejes
M.add_nodes_from([0,1])
M.add_edge(0,1)
M.add_edge(0,1);

A diferencia del grafo dirigido, aca vemos que la relación entre el mismo par de nodos se cuenta tantas veces como la agreguemos con el método `add_edge` o `add_edge_from`.

In [ ]:
# Ahora contamos con más de una conexión entre el mismo par de nodos!
M.number_of_edges(u=0, v=1)

In [ ]:
nx.draw(M, with_labels=True)

Lamentablemente, NetworkX no soporta la representación de ejes paralelos. Más específicamente, el método `draw`. Si bien el grafo puede instanciarse con `múltiples` conexiones, como así también con `selfloops`, los ejes se dibujan como trazas simples (por eso verán que se superponen en el ejemplo anterior). Las funcionalidades para plotear que provee NetworkX son básicas dado que, como se sugiere [desde la misma librería](https://networkx.org/documentation/stable/reference/drawing.html) el principal objetivo es el análisis de grafos antes que si visualización. Para la representación de multigrafos no dirigidos, se pueden explorar otras librerías como [graphviz](https://www.graphviz.org/gallery/) y [pygraphviz](https://pygraphviz.github.io/documentation/stable/install.html), las cuales cuentan ya con [algunos métodos](https://networkx.org/documentation/stable/reference/drawing.html#module-networkx.drawing.nx_agraph) que por el momento son soportados pero que no abordaremos en el presente notebook.

Agreguemos ahora algunas conexiones más para poder comparar nuestro multigrafo con el grafo dirigido que instanciamos previamente:

In [ ]:
# agregamos dos nuevas conexiones, entre 1 y 0 y 1 consigo mismo
M.add_edge(1,0)
M.add_edge(1,1);

In [ ]:
# vemos que a diferencia del grafo dirigido, el multigrafo si contempla los ejes mútliples
M.number_of_edges()

In [ ]:
# sin importar la direccionalidad de la relación
M.number_of_edges(u=0, v=1)

In [ ]:
# por eso, la cantidad de ejes entre 0 y 1 o entre 1 y 0 es la misma
M.number_of_edges(u=1, v=0)

Acá introducimos un nuevo concepto: los `selfloops`. Con `draw` tampoco podemos representarlos en un multigrafo no dirigido. Sin embargo, podemos ver con el método `number_of_selfoops` que el grafo que instanciamos con anterioridad contiene al menos uno. Por el momento sólo presentemos este concepto, en breve ya agregarmos algunos detalles ...

In [ ]:
# los selfloops podemos identificarlos de esta manera
M.number_of_edges(u=1, v=1)

In [ ]:
# o con el método que mencionamos recientemente
nx.number_of_selfloops(M)

#### 1.2.3. Multigrafos dirigidos

Al igual que la anterior, esta clase contempla tanto ejes paralelos como selfloops. Ahora, con la posibilidad de darle direccionalidad a dichas conexiones.Acá el [hipervínculo a la documentación](https://networkx.org/documentation/stable//reference/classes/multidigraph.html).

In [ ]:
MD = nx.MultiDiGraph([(0,1),(0,1),(1,0),(1,1)])
type(MD)

In [ ]:
nx.draw(MD, with_labels=True, arrows=True, connectionstyle='arc3, rad = 0.1')

La representación de multigrafos dirigidos no son una excepción. Lamentablemente, su visualización padece del mismo defecto que los multigrafos dirigidos. Es decir, los ejes múltiples se trazan como simples. Revisemos, sin embargo, cómo son contabilizados.

In [ ]:
# nuestro multigrafo dirigido tiene 4 ejes,
MD.number_of_edges()

In [ ]:
# de los cuales 2 son en dirección 0 -> 1
MD.number_of_edges(u=0, v=1)

In [ ]:
# 1 en dirección inversa (1 -> 0)
MD.number_of_edges(u=1, v=0)

In [ ]:
# y por último un selfloop
MD.number_of_edges(u=1, v=1)

Tal y como vimos con el grafo dirigido, los ejes son representados considerando el orden de los nodos que son conectados. Es decir, un multigrafo dirigido agrega la posibilidad de trabajar con ejes paralelos contemplando direccionalidad (algo que los grafos dirigidos simples no permiten).

### 1.3. Selfloops

Otro aspecto que vimos, algunas clases soportaban y otras no, eran los self-loops. Esto no es otra cosa que un nodo conectándose consigo mismo. Veámos con qué métodos y alternativas contamos para detectar este tipo de conexiones en un grafo ...

In [ ]:
# Como encontrar self loops?

def find_selfloop_node(G):
    '''
    Encontrar nodos con self-loops en un grafo determinado
    '''

    nodes_in_selfloops = []

    # Iterar sobre todos los ejes de G
    for u,v in G.edges():

        # Chequear si el nodo u y el nodo v son iguales:
        if u == v:
            nodes_in_selfloops.append(u)

    return nodes_in_selfloops

In [ ]:
# Usemos un grafo para el que sabemos tenemos self loops
len(find_selfloop_node(MD))

In [ ]:
# Veamos si cuenta lo mismo
nx.number_of_selfloops(MD) == len(find_selfloop_node(MD))

### 1.4. Matrices de adyacencia

Las matrices de adyacencia son un concepto muy útil en grafos, más si lo encaramos desde la perspectiva de redes de transporte. Esencialmente porque este tipo de matrices indican con `1` si existe conexión entre un par de nodos y con `0` para lo contrario. También es posible indicar la intensidad de esa conexión si utilizamos un conjunto de pesos.

In [ ]:
G.edges(data=True) # asi se conectan los nodos de la red

In [ ]:
# Convertimos a G en una matriz (1 indica si hay edge entre nodos)
A = nx.to_numpy_array(G)

In [ ]:
A

Una librería bastante interesante para visualizar grafos es la librería [nxviz](https://github.com/ericmjl/nxviz). Esta permite expandirse un poco más allá de las limitaciones que mencionamos anteriormente. Uno de los principales contribuyentes es el desarrollador de NetworkX, [Eric Ma](https://github.com/ericmjl). Nosotros no vamos a utilizarla pero aquellos interesados pueden explorarla por su cuenta!

In [ ]:
import numpy as np

# Visualizamos la matriz de adyacencia
fig, ax = plt.subplots(figsize=(5, 5))
cax = ax.matshow(A, cmap='YlGnBu')

plt.title("Matriz de adyacencia");

In [ ]:
# Creamos una matriz de adyacencia personalizada usando el atributo 'pasajeros'
n = len(G.nodes)
A = np.zeros((n, n))

for u, v, data in G.edges(data=True):
    A[u, v] = data['pasajeros']
    A[v, u] = data['pasajeros']  # Asumiendo que el grafo es no dirigido

# Graficar la matriz de adyacencia
fig, ax = plt.subplots(figsize=(7, 7))
cax = ax.matshow(A, cmap='YlGnBu')

# Añadir la barra de colores
plt.colorbar(cax)
plt.title("Adjacency Matrix (Pasajeros)")
plt.show()

## Sección 2: Métricas de un grafo

Como vimos, un grafo se compone por nodos y enlaces que los conectan. Al mismo tiempo, entendimos que la manera en la que estos se vinculan define varios tipos de grafos.

Asimismo, es importante remarcar que para describir una red, los grafos cuentan con distintas métricas. Todas ellas, detallan la manera en la que los nodos se encuentran conectados. De allí, la importancia del tipo de grafo que elijamos para representar una red determinada.

En otras palabras, las métricas de una red también se encuentran sujetas al tipo de grafo que hayamos elegido para representarla. No es lo mismo describir relaciones simétricas sin un sentido definido que otras que sí lo tienen. Por lo tanto, seguiremos trabajando bajo la idea de grafos dirigidos y no dirigidos.

Veamos cómo esto incide en la manera de caracterizar cuantitativamente un grafo.

### 2.1. Node degree

Una métrica esencial para describir una red es el [grado de un nodo](https://networkx.org/documentation/stable/reference/classes/generated/networkx.Graph.degree.html). Este puede definirse como el número de ejes adyacentes a un nodo.

También se puede hablar del grado ponderado de un nodo. Si, por ejemplo, nuestra ponderación se apoyara en la cantidad de pasajeros que guardamos como atributo del eje, el grado ponderado sería la suma de todos esos pasajeros (o pesos) que inciden en el nodo en cuestión.

Otra detalle relevante para definir el grado de un nodo es el tipo del grafo. Si trabajamos con direccionalidad, es importante remarcar que el grado será la suma de las conexiones entrantes y salientes, como se puede apreciar a continuación ...

<img align="center" width="1500" height="500" src="https://github.com/PyMap/AUPY/blob/master/imagenes/distribucion_grados.png?raw=1" style="float: center; padding: 0 15px">

In [ ]:
# retomemos nuestro grafo no dirigido inicial
type(G)

In [ ]:
# por ejemplo, el nodo 7 se conecta simétricamente con los nodos 5, 6 y 8
G.nodes(), G.edges()

In [ ]:
# con lo cual sabemos que es un nodo de grado 3
G.degree[7]

In [ ]:
# así también podemos ver los grados de todos los nodos en el grafo
list(G.degree(G.nodes()))

In [ ]:
# si estuviésemos trabajando con un grafo dirigido de izquierda a derecha la relación es un out, y la inversa es un in degree
GD = G.to_directed()
GD.nodes(), GD.edges()

In [ ]:
# por ejemplo, el 7 tiene 3 relaciones salientes y 3 entrantes
GD.degree[7]

In [ ]:
list(GD.degree(GD.nodes()))

### 2.2. Grado promedio de una red

Ahora, no hablaremos de los componentes de la red. Sino, de la red misma. Es decir, el grado promedio es una medida de conectividad de la red que indica, en promedio, con cuántos vecinos conecta un nodo dentro de la red.

<img align="center" width="1500" height="500" src="https://github.com/PyMap/AUPY/blob/master/imagenes/grado_promedio_red.png?raw=1" style="float: center; padding: 0 15px">

Para un grafo no dirigido, podríamos decir que el grado promedio es igual a la suma de los grados de todos sus nodos, dividido por la totalidad de nodos en el grafo. La suma de los grados también podría ser interpretada como la cantidad de ejes multiplicada por dos. Dado que estos son simétricos y, por lo tanto, cuentan más de una vez otorgan un grado a cada uno de los dos nodos: de A a B y de B a A. Veamos un ejemplo.

In [ ]:
# tenemos 8 ejes conectando 9 nodos
G.number_of_edges(), G.number_of_nodes()

In [ ]:
# o, puesto en otros términos
grados = 0
for n in G.nodes():
    grados += G.degree[n]

In [ ]:
# o 16 conexiones o ejes adyacentes entre nuestros 9 nodos
grados

In [ ]:
# Entonces podríamos calcular el grado promedio multiplicando los ejes simétricamente (como en la fórmula)
np.ceil((2*G.number_of_edges())/G.number_of_nodes())

In [ ]:
# o usando la suma de los grados de los nodos, que sería lo mismo
np.ceil(grados/G.number_of_nodes())

En cambio, en un grafo dirigido cada eje o enlace representa 1 grado y no dos (como en el no dirigido). Esto porque otorgan un sólo un grado de entrada o salida al nodo (de A a B o de B a A). Por lo tanto, para un grafo dirigido, el grado promedio es simplemente el número de aristas o ejes dividido por el número de vértices o nodos.

In [ ]:
np.ceil((GD.number_of_edges())/GD.number_of_nodes())

¿Y cómo interpretamos esta métrica? Si bien los grafos que estamos usando no fueron estrictamente pensados para su comparabilidad, el grado promedio podría servir para evaluar cuan conectados entre sí se encuentran los componentes de una red. Por ejemplo, en nuestro grafo no dirigido sabemos que, en promedio, cada nodo conecta con casi otros dos. Obviamente porque en un grafo no dirigido las relaciones son recíprocas. Mientras que en el ejemplo dirigido, sabemos que todos los nodos conectan entre sí.  

### 2.3. Degree centrality

El [degree centrality](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.centrality.degree_centrality.html) es una medida de centralidad del nodo en la red. Si el grado del nodo es la cantidad de ejes adyacentes al mismo, esta métrica no indica otra cosa que ese mismo valor expresado en un ratio 0-1. Esto facilita la lectura. Es decir, permite interpretar rápidamente cuan relevante es un nodo en la red sin tener en cuenta la totalidad de conexiones dentro de la misma. En otras palabras, valores cercanos a 1 indican mayor centralidad dentro de la red.

Esta métrica suele calcularse dividiendo la cantidad de vecinos o grado del nodo por la totalidad de conexiones posibles dentro de la red. Esto es, la cantidad total de ejes dentro del grafo (en el caso que todos los nodos estuviesen conectados con al menos un vecino, es decir ningún nodo aislado) o el total de nodos -1 (es decir, todas las conexiones posibles).

Por lo tanto el grado de centralidad es una medida de relevancia dentro de la red que indica la cantidad de conexiones o vecinos efectivos sobre las que se podría tener. Un `degree centrality` de 0.2 indica que dicho nodo mantiene una conexión efectiva con el 20% del total de conexiones posibles. Veamos cómo calcular esta medida.

In [ ]:
# Volviendo a nuestro grafo no dirigido, podemos ver que el nodo 2 tiene dos vecinos
list(G.neighbors(2))

In [ ]:
# lo que equivale a decir que es un nodo de grado 2
G.degree[2]

In [ ]:
# con una centralidad del 25%, ya sea la calculemos por el total de nodos
G.degree[2]/(G.number_of_nodes()-1)

In [ ]:
# o de ejes ...
G.degree[2]/(G.number_of_edges())

In [ ]:
# Y con el método degree centrality podemos obtener un diccionario con la centralidad de cada nodo
deg_cent = nx.degree_centrality(G)
deg_cent

In [ ]:
# Estaciones por grado de centralidad
node_labels = [G.nodes[i]['estacion'] for i in deg_cent.keys()]
node_colors = [G.nodes[i]['color'] for i in deg_cent.keys()]

fig, ax = plt.subplots(figsize=(12, 4))
nodes = list(deg_cent.keys())
centrality_values = list(deg_cent.values())

ax.bar(node_labels, centrality_values, color=node_colors)
ax.set_xlabel('Estaciones')
ax.set_ylabel('Centralidad de Grado')
ax.grid(True, alpha=0.2)

plt.xticks(rotation=45);

Con un grafo dirigido, no hubiese sido muy diferente. Siempre teniendo en cuenta la diferencia que se agregan las conexiones entrantes y salientes. Por ejemplo, el nodo 2 tiene dos vecinos. Pero se conecta con 1 de manera entrante y saliente. De la misma forma, lo hace con 5. Así, el grado del nodo 2 es de 4 conexiones adyacentes.

In [ ]:
GD.edges()

In [ ]:
# el nodo 2 tiene dos vecinos
list(GD.neighbors(2))

In [ ]:
# entre ambos alcanza un grado de 4 conexiones adyacentes
GD.degree(2)

In [ ]:
# dos entrantes y dos salientes
GD.in_degree(2) + GD.out_degree(2)

In [ ]:
# lo que le otorga una centralidad de 0.5 dentro del grafo
GD.degree[2]/(GD.number_of_nodes()-1)

...y aca todos los ratios de centralidad para nuestro grafo dirigido. Las estaciones 7 y 5 parecerian ser las de mayor centralidad. Es decir, son nodos centrales dentro de la red!

In [ ]:
nx.degree_centrality(GD)

### 2.4. Distancias y búsqueda de caminos

En teoría de grafos, la [distancia](https://es.wikipedia.org/wiki/Distancia_(teor%C3%ADa_de_grafos)) puede ser definida como la cantidad mínima de ejes o aristas que deben recorrerse para conectar dos nodos.

Es posible que al analizar el grado de conectividad dentro de una red nos encontremos frente dos tipos de situaciones: que existe un camino más corto o que los nodos no se encuentran si quiera conectados.

#### 2.4.1.  Path finding

In [ ]:
# Comencemos por recordar el grafo no dirigido que contruimos previamente
nx.draw(G,pos, with_labels=True, font_color='white', node_size=400, node_color=colors)

In [ ]:
# Este contaba con un total de ocho nodos o estaciones
G.nodes

Ahora, armemos una función que nos permita identificar si un mismo par de nodos de la red se encuentra conectado.

In [ ]:
def existe_path_entre_nodos(grafo, nodo1, nodo2):
    """
    Verifica si dos nodos de un grafo se encuentran conectados.
    """

    # inicializamos una lista con el extremo inicial de cada path o camino que vamos a recorrer
    cabeceras = [nodo1]

    # creamos un contador para almacenar la cantidad de nodos intermedios
    distancia = 0

    # iteramos la lista de nodos cabecera que almacenamos previamente
    for nodo in cabeceras:

        # Obtenemos los vecinos de cada nodo cabecera
        vecinos = grafo.neighbors(nodo)

        # y verificamos si el nodo de destino se encuentra dentro de los vecinos del nodo cabecera
        if nodo2 in vecinos:
            print('Existe un camino adyacente entre los nodos {} y {}'.format(nodo1, nodo2))
            # agregamos un nodo de distancia por conexión adyacente
            distancia += 1
            return distancia

        else:
            print('No existe adyacencia entre nodos {} y {}'.format(nodo1, nodo2))
            return distancia

Vamos a aplicar esta función para evaluar la presencia de caminos adyacentes en alguna de las líneas de subte que creamos en nuestro grafo no dirigido. Si bien puede parecer algo evidente, veamos qué nos devuelve esta función.

In [ ]:
# utilizamos los nodos 0,1 y 2
nodos_b = list(G.nodes)[:2]
for i in nodos_b:
    d = existe_path_entre_nodos(grafo=G, nodo1=i, nodo2=i+1)

c = d * len(nodos_b)
a = nodos_b[0] # nodo inicial
b = nodos_b[-1]+1 # nodo final
print(f'Los estaciones {a} y {b} están separadas por {d} nodo de distancia y suman {c} conexiones adyacentes')

Nuestra función sólo contempla conexiones adyacentes dentro de una misma línea.  

In [ ]:
# Por eso, no reconoce conexiones adyacentes entre el último nodo de la línea B y el primero de la D ...
existe_path_entre_nodos(grafo=G, nodo1=2, nodo2=3)

...no significa que ambas líneas no se encuentren conectadas. La función que acabamos de crear nos indica si un par de nodos son adyacentes o no, algo similar a lo que vimos con las matrices pero de una manera programática bastante diferente. Iterando en un for, este método nos permite identificar si existen caminos intermedios entre dos vértices ordenados.

In [ ]:
# Como, por ejemplo, el 5 y el 2
existe_path_entre_nodos(grafo=G, nodo1=5, nodo2=2)

En definitiva, lo que acabamos de hacer es identificar caminos intermedios que luego vamos a sumar para establecer la distancia total entre dos extremos de una misma red.

#### 2.4.2.  Shortest path

El `shortest path` o [camino mas corto](https://es.wikipedia.org/wiki/Problema_del_camino_m%C3%A1s_corto#:~:text=En%20la%20teor%C3%ADa%20de%20grafos,a%20otra%20en%20un%20mapa.) implica, en teoría de grafos, encontrar el conjunto de ejes que minimice la suma de pesos entre dos extremos de una red. Es decir, entre un par de nodos determinado.

Si nos retrotraemos al inicio del notebook, recordarán que un valor que definimos para ponderar la relación entre nodos (a partir de sus ejes) fue la cantidad de pasajeros. Sin embargo, en el ejemplo anterior, no utilizamos dicha medida para la suma de distancias.

Esto es, porque como dijimos previamente, esta métrica puede calcularse en contextos de relaciones ponderadas y no ponderadas. Comúnmente, en redes de transporte, la estimación del `camino más corto` suele trabajarse con ejes ponderados por tiempo de viaje o por distancia lineal. Lo que nosotros hicimos anteriormente fue establecer el largo de un camino sumando conexiones adyadentes con un mismo peso. Pero esto se puede hacer de una manera mucho más eficiente.

`NetworkX` cuenta con [algunos métodos](https://networkx.org/documentation/stable/reference/algorithms/shortest_paths.html) para determinar el camino más corto para ejes ponderados como sin ponderar. Entre los más importantes para tener en cuenta:

`all_pairs_shortest_path`: calcula el camino más corto entre todos los pares nodos de un grafo no ponderado

`all_pairs_shortest_path_length`: calcula el largo del camino más corto entre todos los pares de nodos de un grafo no ponderado.

`all_pairs_dijkstra_path`: calcula el camino más corto entro todos los pares de nodos de un grafo ponderado

`all_pairs_dijkstra_path_length`: calculata el largo del camino más corto entre todos los pares de nodos de un grafo ponderado.


Otra alternativa que ofrece `NetworkX` es trabajar de manera [genérica](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.generic.shortest_path.html#networkx.algorithms.shortest_paths.generic.shortest_path) tanto para determinar el path como [su largo](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.generic.shortest_path_length.html#networkx.algorithms.shortest_paths.generic.shortest_path_length). Es decir, pasar el peso de los ejes como un parámetro más.

##### 2.4.2.1  Unweighted

Retomemos el ejemplo anterior para ver cómo funciona el camino más corto en grafos no ponderados

In [ ]:
# para llegar de 0 a 2, el camino más corto implica pasar por 1
nx.shortest_path(G, source=0, target=2)

In [ ]:
# lo que equivale a la sumatoria de dos caminos adyacentes
nx.shortest_path_length(G, source=0, target=2)

Como dijimos, el cálculo del camino más corto implica la identificación de conexiones adyacentes. Ahora bien, estos métodos nos permiten determinarlo de una manera mucho más eficiente, ya que por ejemplo, podemos hacerlo entre todos los nodos de la red...

In [ ]:
# calculamos el camino más corto entre todos los pares de nodos de nuestra red
caminos=dict(nx.all_pairs_shortest_path(G))

In [ ]:
caminos

In [ ]:
# lo que nos permite obtener el path más corto entre cualquier par origen - destino
caminos[0][2]

In [ ]:
# y tambien el largo del path
largo_caminos=dict(nx.all_pairs_shortest_path_length(G))

In [ ]:
# al que podemos acceder de la misma manera. Ubicando primero la key de origen y luego la de destino
largo_caminos[0][2]

Calculando el `shortest path` podemos encontrar cuáles son los nodos intermedios de un camino sin importar si estos son adyacentes. Por ejemplo, si queremos ir del primer nodo (línea B) al tercero (línea D):

In [ ]:
# aca no importa si estamos en la misma línea
nx.shortest_path(G, source=0, target=3)

esto nos permite ver que si bien, como marcamos con anterioridad, el último nodo de la línea B y el primero de la D no se encuentran conectados de manera adyacente, uno podría llegar de un extremo al otro recorriendo ...

In [ ]:
# un total de 5 tramos!
nx.shortest_path_length(G, source=0, target=3)

Esto, gracias a que existe una conexión adyacente entre:

In [ ]:
# los nodos 5 y 2!
existe_path_entre_nodos(grafo=G, nodo1=5, nodo2=2)

##### 2.4.2.2  Weighted

En redes de transporte es muy común que la estimación del camino más corto contemple ponderadores como tiempo y distancia. Ahora, veremos cómo especificar atributos de los ejes en los métodos detallados para calcular el `shortest path` con grafos ponderados.

Esto lo vamos a hacer principalmente para ilustrar cómo utilizar dichos métodos. **Dado que estamos trabajando con un recorte de la red de subterráneos, y que la conexión entre líneas siempre se da entre el mismo par de nodos adyacentes, los resultados van a ser iguales a una estimación no ponderada.**

In [ ]:
# recordemos nuestroa atributos
G.edges(data=True)

In [ ]:
# y agreguemos algún ponderador que tenga más sentido que nuestro valor de pasajeros
def agrega_atributos_ejes():
    conexiones_lineas = [(2,5),(5,7)]
    conexiones_estaciones = [i for i in G.edges if i not in conexiones_lineas]

    for i in conexiones_estaciones:
        G.edges[i[0], i[1]]['tiempo_viaje']=random.randint(1,5)

    for i in conexiones_lineas:
        G.edges[i[0], i[1]]['tiempo_viaje']=random.randint(1,60)

In [ ]:
agrega_atributos_ejes()

In [ ]:
nx.shortest_path(G, source=0, target=3, weight='tiempo_viaje')

In [ ]:
caminos_ponderados = dict(nx.all_pairs_dijkstra_path(G, weight='tiempo_viaje'))

In [ ]:
caminos_ponderados[0][3]

El uso es similar. Uno podría calcular caminos entre pares origen destino, o para toda la red. Obteniendo luego el resultado deseado una vez que se llama el par buscado en el objeto instanciado dentro de un diccionario. Esto aplica tanto para el algoritmo `dijkstra` que devuelve el path mismo como su largo.

### 2.5. Betweenness centrality

Todo este preludio con una única finalidad. Poder comprender otra de las métricas más utilizadas para estudiar `centralidad` dentro de una red. El `betweenness centrality` es una métrica de intermediación que permite cuantificar el número de veces que un nodo actúa como puente a lo largo del camino más corto entro dos nodos.

En otras palabras, es una medida que capta la importancia de un nodo dentro de una red. Esto, a partir de su presencia en la totalidad de tramos o caminos más cortos que existen dentro de la misma.

A mayor frecuencia de apariciones de un nodo en los caminos más cortos, es de esperar que este cumpla un rol central en el flujo de información de un sector de la red a otro.  

Técnicamente, [esta métrica](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.centrality.betweenness_centrality.html#networkx.algorithms.centrality.betweenness_centrality) devuelve el porcentaje de caminos más cortos que deben pasar por un nodo específico. A mayor cantidad de caminos, más alto será el valor de intermediación del nodo!

In [ ]:
# Calculemos el valor de intermediación para nuestra red de subterráneos
nx.betweenness_centrality(G)

In [ ]:
# estas eran nuestras labels con el id del nodo
lab

In [ ]:
# instanciamos la métrica
intermediacion = nx.betweenness_centrality(G)

In [ ]:
# y obtenemos la key con valor más alto
max(intermediacion, key=intermediacion.get)

En nuestro grafo, la estación con mayor valor de intermediación es `9 de Julio`. Algo esperable si visualizamos que es una estación que permite conectar distintas líneas del subterráneo entre sí. El índice de intermediación (o `betweenness centrality` nos indica que el 75% de los `shortest path` pasan por este nodo!